## Prediktivno modelovanje 

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/final.csv")
print(df.shape)
df.head()


(25562, 27)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avg_days_before_deadline_CMA,avg_days_before_deadline_TMA,avg_score_CMA,avg_score_TMA,count_submitted_CMA,count_submitted_TMA,std_score_CMA,std_score_TMA,total_assessments_CMA,total_assessments_TMA,submission_rate_TMA,submission_rate_CMA,total_clicks,last_active_day,unique_activities
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,0.0,1.0,0.0,81.5,0.0,2.0,0.0,4.949747,0.0,2.0,1.0,0.0,587.0,87.0,6.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,0.0,-0.5,0.0,69.0,0.0,2.0,0.0,1.414214,0.0,2.0,1.0,0.0,821.0,90.0,6.0
2,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,0.0,2.5,0.0,71.5,0.0,2.0,0.0,0.707107,0.0,2.0,1.0,0.0,992.0,90.0,7.0
3,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,0.0,-14.0,0.0,49.5,0.0,2.0,0.0,27.577164,0.0,2.0,1.0,0.0,684.0,90.0,7.0
4,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,0,60,N,Pass,0.0,-5.0,0.0,74.0,0.0,2.0,0.0,7.071068,0.0,2.0,1.0,0.0,1060.0,86.0,7.0


In [2]:
df["final_result"].value_counts()

final_result
Pass           12361
Fail            7044
Withdrawn       3133
Distinction     3024
Name: count, dtype: int64

In [3]:
df["target"] = df["final_result"].map({
    "Pass": 0,
    "Distinction": 0,
    "Fail": 1,
    "Withdrawn": 1
})
df['target'].value_counts()

target
0    15385
1    10177
Name: count, dtype: int64

In [4]:
X = df.drop(columns=["final_result","target"])
y = df["target"]

In [5]:
X = X.drop(columns=["id_student"])

In [6]:
df["year"] = df["code_presentation"].astype(str).str[:4]

df["year"].value_counts()

year
2014    14507
2013    11055
Name: count, dtype: int64

In [7]:
train_df = df[df["year"] == "2013"].copy()
test_df = df[df["year"] == "2014"].copy()

In [8]:
X_train = train_df.drop(columns=["final_result", "target", "id_student", "year"], errors="ignore")
y_train = train_df["target"]

X_test = test_df.drop(columns=["final_result", "target", "id_student", "year"], errors="ignore")
y_test = test_df["target"]

In [9]:
print("Train skup:", X_train.shape)
print("Test skup:", X_test.shape)

print("Train target:")
print(y_train.value_counts(normalize=True))

print("Test target:")
print(y_test.value_counts(normalize=True))

Train skup: (11055, 25)
Test skup: (14507, 25)
Train target:
target
0    0.594301
1    0.405699
Name: proportion, dtype: float64
Test target:
target
0    0.607638
1    0.392362
Name: proportion, dtype: float64


In [10]:
num_col = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_col = X_train.select_dtypes(include=["object"]).columns
print("Numericke kolone:",num_col)
print("Kategorijske kolone:",cat_col)

Numericke kolone: Index(['num_of_prev_attempts', 'studied_credits',
       'avg_days_before_deadline_CMA', 'avg_days_before_deadline_TMA',
       'avg_score_CMA', 'avg_score_TMA', 'count_submitted_CMA',
       'count_submitted_TMA', 'std_score_CMA', 'std_score_TMA',
       'total_assessments_CMA', 'total_assessments_TMA', 'submission_rate_TMA',
       'submission_rate_CMA', 'total_clicks', 'last_active_day',
       'unique_activities'],
      dtype='str')
Kategorijske kolone: Index(['code_module', 'code_presentation', 'gender', 'region',
       'highest_education', 'imd_band', 'age_band', 'disability'],
      dtype='str')


C:\Users\Dejana-NB\AppData\Local\Temp\ipykernel_10100\1429236844.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_col = X_train.select_dtypes(include=["object"]).columns


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_col),
    ("cat", categorical_transformer, cat_col)
])

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
results = []
cv=StratifiedKFold(n_splits=5,shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

for name, model in models.items():
    
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    results.append({
        "model": name,
        "accuracy": scores["test_accuracy"].mean(),
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
        "roc_auc": scores["test_roc_auc"].mean()
    })
    

results_df = pd.DataFrame(results)
results_df.sort_values("roc_auc", ascending=False)

,model,accuracy,precision,recall,f1,roc_auc
4,Gradient Boosting,0.800543,0.815652,0.657302,0.727756,0.868395
3,Random Forest,0.795115,0.807967,0.650167,0.720233,0.860680
5,AdaBoost,0.791407,0.787502,0.665998,0.721282,0.858756
0,Logistic Regression,0.795115,0.813628,0.642809,0.717947,0.857322
1,KNN,0.761646,0.766980,0.593088,0.668599,0.796622
2,Decision Tree,0.714790,0.645618,0.658417,0.651925,0.705845


## Optimizacija

### RandomizedSearchCV

In [14]:
from sklearn.model_selection import RandomizedSearchCV

gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

In [15]:
param_dist = {
    "model__n_estimators": [50, 100, 150, 200],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

In [21]:
random_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': [0.01, 0.05, ...], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation st

In [22]:
print("Najbolji parametri:")
print(random_search.best_params_)
print("Najbolji ROC AUC na validaciji:")
print(random_search.best_score_)

Najbolji parametri:
{'model__n_estimators': 150, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_depth': 4, 'model__learning_rate': 0.05}
Najbolji ROC AUC na validaciji:
0.8680187128072774


In [23]:
best_random_model = random_search.best_estimator_

### GridSearchCV

In [17]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [50, 100],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 4]
}

In [24]:
grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 4], 'model__n_estimators': [50, 100]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and

In [25]:
print("Najbolji parametri:")
print(grid_search.best_params_)

print("Najbolji ROC AUC na validaciji:")
print(grid_search.best_score_)

Najbolji parametri:
{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Najbolji ROC AUC na validaciji:
0.8683953275674539


In [26]:
best_grid_model = grid_search.best_estimator_

### Bayesian

In [19]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [27]:
bayes_search = BayesSearchCV(
    estimator=gb_pipeline,
    search_spaces={
        "model__n_estimators": Integer(50, 200),
        "model__learning_rate": Real(0.01, 0.2, prior="log-uniform"),
        "model__max_depth": Integer(2, 5),
        "model__min_samples_split": Integer(2, 10),
        "model__min_samples_leaf": Integer(1, 4)
    },
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

bayes_search.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,search_spaces,"{'model__learning_rate': Real(low=0.01...m='normalize'), 'model__max_depth': Integer(low=2...m='normalize'), 'model__min_samples_leaf': Integer(low=1...m='normalize'), 'model__min_samples_split': Integer(low=2...m='normalize'), ...}"
,optimizer_kwargs,None
,n_iter,10
,scoring,'roc_auc'
,fit_params,None
,n_jobs,-1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,StratifiedKFo... shuffle=True)


In [28]:
print("Najbolji parametri:")
print(bayes_search.best_params_)

print("Najbolji ROC AUC na validaciji:")
print(bayes_search.best_score_)

Najbolji parametri:
OrderedDict({'model__learning_rate': 0.03416312199509112, 'model__max_depth': 4, 'model__min_samples_leaf': 4, 'model__min_samples_split': 5, 'model__n_estimators': 151})
Najbolji ROC AUC na validaciji:
0.8675777876194791


In [29]:
best_bayes_model = bayes_search.best_estimator_

### Rezultati

In [30]:
rez_optimizacije = pd.DataFrame([
    {
        "method": "RandomizedSearchCV",
        "cv_roc_auc": random_search.best_score_
    },
    {
        "method": "GridSearchCV",
        "cv_roc_auc": grid_search.best_score_
    },
    {
        "method": "BayesSearchCV",
        "cv_roc_auc": bayes_search.best_score_
    }
])

rez_optimizacije.sort_values("cv_roc_auc", ascending=False)

,method,cv_roc_auc
1,GridSearchCV,0.868395
0,RandomizedSearchCV,0.868019
2,BayesSearchCV,0.867578


In [31]:
final_model = grid_search.best_estimator_
print("Finalni model: GridSearchCV")
print("CV ROC-AUC:", grid_search.best_score_)
print("Najbolji parametri:", grid_search.best_params_)

Finalni model: GridSearchCV
CV ROC-AUC: 0.8683953275674539
Najbolji parametri: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}


### Threshold analiza (rad sa nebalansiranim klasama)

In [34]:
from sklearn.model_selection import cross_val_predict

y_proba_oof=cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:,1]

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

threshold_rez = []

for t in thresholds:
    y_pred_threshold = (y_proba_oof >= t).astype(int)
    
    threshold_rez.append({
        "threshold": t,
        "accuracy": accuracy_score(y_train, y_pred_threshold),
        "precision": precision_score(y_train, y_pred_threshold),
        "recall": recall_score(y_train, y_pred_threshold),
        "f1": f1_score(y_train, y_pred_threshold)
    })

threshold_rez_df = pd.DataFrame(threshold_rez)
threshold_rez_df

,threshold,accuracy,precision,recall,f1
0,0.3,0.771054,0.680725,0.820513,0.744111
1,0.4,0.795929,0.755093,0.735563,0.745200
2,0.5,0.800543,0.815265,0.657302,0.727811
3,0.6,0.795206,0.864936,0.586845,0.699256
4,0.7,0.779557,0.905063,0.510145,0.652502


## evaluacija na test skupu

In [36]:
final_model.fit(X_train, y_train)
best_threshold=0.4
y_proba_test = final_model.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= best_threshold).astype(int)

final_test_results = {
    "accuracy": accuracy_score(y_test, y_pred_test),
    "precision": precision_score(y_test, y_pred_test),
    "recall": recall_score(y_test, y_pred_test),
    "f1": f1_score(y_test, y_pred_test),
    "roc_auc": roc_auc_score(y_test, y_proba_test)
}
final_test_results

{'accuracy': 0.7607361963190185,
 'precision': 0.6885719137374766,
 'recall': 0.7124033731553057,
 'f1': 0.7002849494862274,
 'roc_auc': 0.8280356265214257}

### Čuvanje modela 

In [37]:
import joblib
joblib.dump(final_model, "../models/final_model.pkl")

['../models/final_model.pkl']

In [38]:
joblib.dump(best_threshold, "../models/threshold.pkl")

['../models/threshold.pkl']